# STAT 301 Individual Assignment 2
#### Prepared by: Maggie Wang, Group 32 (Daniel Wick, Emmy Yu Ting Wong, Zirun Xu)

In [15]:
devtools::install_github("diverse-data-hub/diversedata")

Skipping install of 'diversedata' from a github remote, the SHA1 (4d9ebacf) has not changed since last install.
  Use `force = TRUE` to force installation



In [16]:
suppressPackageStartupMessages({
    library(diversedata)
    library(dplyr)
    library(ggplot2)
    library(broom)
    library(tidyverse)
    library(modelr)
    library(yardstick)
    library(leaps)
    library(MASS)
    library(car)
})
data("wildfire")

## Methods and Plan

### Part A
My research question from assignment one was: How do environmental conditions effect the fire spread rate? Based on feedback from Assignment 1 and discussion with my groupmates, I've revised the research question to be slightly more specific.

Revised question: Given that a fire is actively spreading, how are environmental conditions associated with the spread rate of a wildfire?

### Part B

Using the `wildfire` dataset, I could fit a multiple linear regression (MLR) model with interaction terms using the variables `weather_conditions_over_fire`, `temperature`, `relative_humidity`, `fuel_type`, `wind_speed`, `fire_position_on_slope` and `wind_direction`. I believe that the only interaction terms that may be significant are between `wind_speed`, `fire_position_on_slope` and `wind_direction`. Increased wind speeds could increase the spread rate of the fire, but the magnitude of the effect could be associated with the position of the fire on a slope, and the direction that the wind is blowing in. This is appropriate as I want to investigate the association of these environmental conditions with the continuous response, `fire_spread_rate`. A stepwise method (`stepAIC()`) will be used for variable selection. I will check for multicollinearity in the model using GVIF.

This method produces interpretable results, so it is easy to understand what factors affect fire behavior.

### Part C
Assumptions:
- Linear relationship between `fire_spread_rate` and predictors.
- Other variables such as `activity_class`, `general_cause`, `fire_type`, and `bucketing_on_fire` are not associated with the response.

Limitations:
- Cannot handle non-linear relationships or multicollinearity.
- Categorical variables have many factors, resulting in a lot of terms.

##  Computational Code and Output

In [18]:
wildfire_clean <- wildfire %>%
    dplyr::select(fire_spread_rate, weather_conditions_over_fire, temperature, relative_humidity, fuel_type, wind_speed, fire_position_on_slope, wind_direction) %>%
    filter(!if_any(everything(), ~ . == "Unknown"))

In [19]:
head(wildfire_clean)

fire_spread_rate,weather_conditions_over_fire,temperature,relative_humidity,fuel_type,wind_speed,fire_position_on_slope,wind_direction
<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0,Clear,18,10,O1a,2,Flat,SW
0,Clear,12,22,O1a,10,Lower 1/3,SW
0,Clear,12,22,O1a,10,Bottom,SW
0,Clear,12,22,O1b,10,Flat,SW
0,Clear,11,32,O1b,20,Flat,S
0,Cloudy,11,25,O1b,10,Flat,W


In [20]:
set.seed(2026)

wildfire_train <- wildfire_clean |>
  group_by(weather_conditions_over_fire, fuel_type, fire_position_on_slope, wind_direction) |>
  sample_frac(0.70) |>
  ungroup()

wildfire_test <- wildfire_clean |>
  anti_join(wildfire_train)

Joining with `by = join_by(fire_spread_rate, weather_conditions_over_fire,
temperature, relative_humidity, fuel_type, wind_speed, fire_position_on_slope,
wind_direction)`


In [ ]:
wildfire_full <- lm(fire_spread_rate ~ weather_conditions_over_fire + temperature + relative_humidity + fuel_type + 
                    wind_speed * fire_position_on_slope * wind_direction, 
                    data = wildfire_train)

In [ ]:
full_model_results <- 
   tidy(wildfire_full, conf.int = TRUE) %>%
   mutate_if(is.numeric, round, 2)

full_model_results

In [ ]:
wildfire_test <- wildfire_test |> 
    add_predictions(wildfire_full, var = "pred_full_OLS")

wildfire_test_metrics <- 
    wildfire_test %>%
    metrics(truth = fire_spread_rate, estimate = pred_full_OLS)

results_plot <- plot(x = wildfire_test$pred_full_OLS, y = wildfire_test$fire_spread_rate)

wildfire_test_metrics

In [ ]:
wildfire_null <- lm(fire_spread_rate ~ 1, data = wildfire_clean)

In [ ]:
n <- nrow(wildfire_train)

modAIC_back <- MASS::stepAIC(wildfire_full, direction = "backwards", 
                             scope = list(lower = wildfire_null, upper = wildfire_full),
                             k = log(n))

summary(modAIC_back)

In [13]:
vif(wildfire_full)

ERROR: Error in eval(expr, envir, enclos): object 'wildfire_full' not found
